# GAL-MAD Model Implementation (VERSION 2)
- Stride/Shift = 16 timesteps
- Window size = 24 timesteps
- Any timestep with label 1 in window -> Window is anomalous

In [ ]:
!pip install torch torchvision torchaudio
!pip install torch_geometric shap pandas scikit-learn numpy

## 1. Constants & Definitions

In [ ]:

services = ['payment', 'shipping', 'redis', 'mongodb', 'dispatch', 'rabbitmq', 'user', 'mysql', 'catalogue', 'ratings', 'web', 'cart']
anomalies = ['rt-delay-catalogue',
 'packetloss-user',
 'low-bandwidth-user',
 'high-cpu-dispatch',
 'high-latency-user-2',
 'high-load-1500',
 'out-of-order-packets-user-2',
 'high-latency-user',
 'service-down-payment',
 'out-of-order-packets-user',
 'packetloss-user-2',
 'high-fileIO-payment',
 'memory-leak-cart',
  'low-bandwidth-user-2']


cumulative_cols = ['container_cpu_system_seconds_total',
'container_cpu_usage_seconds_total',
'container_cpu_user_seconds_total',
'container_network_receive_bytes_total',
'container_network_receive_errors_total',
'container_network_receive_packets_dropped_total',
'container_network_receive_packets_total',
'container_network_transmit_bytes_total',
'container_network_transmit_errors_total'	,
'container_network_transmit_packets_dropped_total',
'container_network_transmit_packets_total',
'container_fs_io_time_seconds_total',
'container_memory_failures_total',
'container_memory_failcnt',
'container_fs_write_seconds_total']

other_cols = ['container_fs_usage_bytes',
'container_memory_rss',
'container_memory_usage_bytes',
'container_memory_working_set_bytes']

cols = other_cols + cumulative_cols

containers = ['payment', 'shipping', 'redis', 'mongo', 'dispatch', 'rabbitmq', 'user', 'mysql', 'catalogue', 'ratings', 'web', 'cart']
metrics = ['rt_ratings_put_PDO_count', 'rt_ratings_put_PDO_sum', 'rt_payment_delete_cart_count', 'rt_payment_delete_cart_sum', 'rt_cart_get_catalogue_count', 'rt_cart_get_catalogue_sum', 'rt_ratings_get_catalogue_count', 'rt_ratings_get_catalogue_sum', 'rt_catalogue_get_mongo_categories_count', 'rt_catalogue_get_mongo_categories_sum', 'rt_catalogue_get_mongo_products_count', 'rt_catalogue_get_mongo_products_sum', 'rt_catalogue_get_mongo_productscat_count', 'rt_catalogue_get_mongo_productscat_sum', 'rt_catalogue_get_mongo_productsku_count', 'rt_catalogue_get_mongo_productsku_sum', 'rt_catalogue_get_mongo_search_count', 'rt_catalogue_get_mongo_search_sum', 'rt_user_get_mongo_checkid_count', 'rt_user_get_mongo_checkid_sum', 'rt_user_get_mongo_history_count', 'rt_user_get_mongo_history_sum', 'rt_user_get_mongo_users_count', 'rt_user_get_mongo_users_sum', 'rt_user_post_mongo_login_count', 'rt_user_post_mongo_login_sum', 'rt_user_post_mongo_order_count', 'rt_user_post_mongo_order_sum', 'rt_user_post_mongo_register_count', 'rt_user_post_mongo_register_sum', 'rt_web_post_payment_count', 'rt_web_post_payment_sum', 'rt_dispatch_get_rabbitmq_count', 'rt_dispatch_get_rabbitmq_sum', 'rt_web_get_ratings_count', 'rt_web_get_ratings_sum', 'rt_cart_delete_redis_count', 'rt_cart_delete_redis_sum', 'rt_cart_get_redis_count', 'rt_cart_get_redis_sum', 'rt_cart_post_redis_count', 'rt_cart_post_redis_sum', 'rt_user_get_redis_count', 'rt_user_get_redis_sum', 'rt_web_get_shipping_calcid_seconds_count', 'rt_web_get_shipping_calcid_seconds_sum', 'rt_web_get_shipping_code_seconds_count', 'rt_web_get_shipping_code_seconds_sum', 'rt_web_get_shipping_postconfirm_seconds_count', 'rt_web_get_shipping_postconfirm_seconds_sum', 'rt_payment_get_user_count', 'rt_payment_get_user_sum', 'rt_payment_post_user_count', 'rt_payment_post_user_sum']


## 2. Dataset and Preprocessing

In [ ]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

class RSAnomicDataset(Dataset):
    def __init__(self, data_path, seq_len=24, stride=16, is_train=True, scaler=None):
        self.seq_len = seq_len
        self.stride = stride
        self.is_train = is_train
        
        raw_data = torch.load(data_path) 
        L = raw_data.shape[0]
        num_nodes = raw_data.shape[1]
        
        if is_train:
            features = raw_data
            self.labels = None
        else:
            features = raw_data[:, :, :20]
            self.labels = raw_data[:, :, 20].numpy()
            
        features_np = features.numpy()
        
        ma_features = np.zeros((L, num_nodes, 2))
        for n in range(num_nodes):
            rt_series = pd.Series(features_np[:, n, 19])
            ma_24 = rt_series.rolling(window=24, min_periods=1).mean()
            ma_features[:, n, 0] = rt_series - ma_24
            ma_240 = rt_series.rolling(window=240, min_periods=1).mean()
            ma_features[:, n, 1] = rt_series - ma_240
            
        features_np = np.concatenate([features_np, ma_features], axis=-1)
        features_flat = features_np.reshape(-1, 22)
        
        if is_train:
            if scaler is None:
                self.scaler = StandardScaler()
                features_flat = self.scaler.fit_transform(features_flat)
            else:
                self.scaler = scaler
                features_flat = self.scaler.transform(features_flat)
        else:
            self.scaler = scaler
            if self.scaler is None:
                raise ValueError("Scaler must be provided for test data")
            features_flat = self.scaler.transform(features_flat)
            
        self.data = features_flat.reshape(L, num_nodes, 22)
        self.data = torch.tensor(self.data, dtype=torch.float32)
        
        if self.labels is not None:
            self.labels = torch.tensor(self.labels, dtype=torch.float32)
            
        self.num_samples = (L - self.seq_len) // self.stride + 1

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        start_idx = idx * self.stride
        x = self.data[start_idx : start_idx + self.seq_len]
        
        if self.labels is not None:
            window_labels = self.labels[start_idx : start_idx + self.seq_len]
            # Nếu chỉ cần 1 timestep lỗi trong 24 timestep, coi như lỗi cả cửa sổ đó
            # max(dim=0)[0] sẽ lấy giá trị lớn nhất theo chiều thời gian (0 hoặc 1)
            y = window_labels.max(dim=0)[0]
            return x, y
        
        return x


## 2.5. Visualize Raw Data
Biểu đồ dưới đây trực quan hóa 20 features gốc (trước khi tính Moving Average) của vi dịch vụ `shipping` (node index = 1) trên cả tập Train và Test.

In [ ]:
import torch
import matplotlib.pyplot as plt
import os

# Cập nhật đường dẫn file cho phù hợp với môi trường Colab/Local của bạn
train_path = 'train/train.pt' if os.path.exists('train/train.pt') else 'E:/VHT/VDT/AD/train/train.pt'
test_path = 'test/test5.pt' if os.path.exists('test/test5.pt') else 'E:/VHT/VDT/AD/test/test5.pt'

try:
    train_raw = torch.load(train_path)
    test_raw = torch.load(test_path)

    # Chọn microservice 'shipping' (index 1)
    node_idx = 1
    ms_name = 'shipping'

    # Lấy 20 features gốc (index từ 0 đến 19)
    train_feats = train_raw[:, node_idx, :20].numpy()
    test_feats = test_raw[:, node_idx, :20].numpy()

    fig, axs = plt.subplots(2, 1, figsize=(15, 10))

    # Plot Train
    for i in range(20):
        axs[0].plot(train_feats[:, i], label=f'Feature {i}', alpha=0.7)
    axs[0].set_title(f'Train Data - 20 Raw Features ({ms_name})')
    axs[0].set_xlabel('Timesteps')
    axs[0].set_ylabel('Value')
    
    # Plot Test
    for i in range(20):
        axs[1].plot(test_feats[:, i], label=f'Feature {i}', alpha=0.7)
    axs[1].set_title(f'Test Data - 20 Raw Features ({ms_name})')
    axs[1].set_xlabel('Timesteps')
    axs[1].set_ylabel('Value')

    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print("File không tồn tại. Vui lòng kiểm tra lại đường dẫn file train.pt và test5.pt")


## 3. GAL-MAD Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv

# Define edges:
# payment(0), shipping(1), redis(2), mongodb(3), dispatch(4), rabbitmq(5),
# user(6), mysql(7), catalog(8), ratings(9), web(10), and cart(11)
source_nodes = [10, 10, 10, 10, 10, 10, 11, 11, 6, 6, 8, 0, 0, 0, 5, 1, 9]
target_nodes = [11, 8, 9, 1, 0, 6, 2, 8, 2, 3, 3, 6, 5, 11, 4, 7, 7]
# Create edge_index correctly
EDGE_INDEX = torch.tensor([source_nodes, target_nodes], dtype=torch.long)

class Encoder(nn.Module):
    def __init__(self, in_channels=22, hidden_dim=32):
        super().__init__()
        self.gat1 = GATConv(in_channels, 32)
        self.gat2 = GATConv(32, 16)
        
        self.lstm = nn.LSTM(
            input_size=16, 
            hidden_size=hidden_dim, 
            num_layers=2, 
            batch_first=True, 
            bidirectional=True
        )
        self.linear = nn.Linear(2 * hidden_dim, 1)
        
    def forward(self, x, edge_index):
        # x: (batch_size, seq_len, num_nodes, in_channels)
        batch_size, seq_len, num_nodes, in_channels = x.shape
        
        num_graphs = batch_size * seq_len
        x_flat = x.view(num_graphs * num_nodes, in_channels)
        
        # Batch edge index for num_graphs
        offsets = torch.arange(num_graphs, device=x.device) * num_nodes
        edge_index_batched = edge_index.unsqueeze(2) + offsets.view(1, 1, -1)
        edge_index_batched = edge_index_batched.view(2, -1)
        
        # Apply GAT
        x_gat = F.relu(self.gat1(x_flat, edge_index_batched))
        x_gat = F.relu(self.gat2(x_gat, edge_index_batched)) # (B * seq * N, 16)
        
        # Reshape for LSTM: (batch * num_nodes, seq_len, 16)
        x_gat = x_gat.view(batch_size, seq_len, num_nodes, 16)
        x_gat = x_gat.transpose(1, 2).contiguous() # (batch, num_nodes, seq_len, 16)
        x_lstm_in = x_gat.view(batch_size * num_nodes, seq_len, 16)
        
        # Apply Bi-LSTM
        out_lstm, (hn, cn) = self.lstm(x_lstm_in)
        
        # Extract final hidden state properly from a 2-layer Bi-LSTM
        # hn shape: (4, batch * num_nodes, hidden_dim)
        final_hidden = torch.cat([hn[-2], hn[-1]], dim=-1) # (batch * num_nodes, 2 * hidden_dim)
        
        # Map to d_z = 1
        z = self.linear(final_hidden) # (batch * num_nodes, 1)
        
        # Reshape to stack embeddings
        z = z.view(batch_size, num_nodes, 1)
        return z

class Decoder(nn.Module):
    def __init__(self, out_channels=22, hidden_dim=32):
        super().__init__()
        # One-to-many LSTM, input size 1 (from d_z)
        self.lstm = nn.LSTM(
            input_size=1, 
            hidden_size=hidden_dim, 
            num_layers=1, 
            batch_first=True, 
            bidirectional=True
        )
        self.linear = nn.Linear(2 * hidden_dim, 16)
        
        self.gat1 = GATConv(16, 32)
        self.gat2 = GATConv(32, out_channels)
        
    def forward(self, z, seq_len, edge_index):
        # z: (batch_size, num_nodes, 1)
        batch_size, num_nodes, _ = z.shape
        
        # One-to-many: repeat the latent vector seq_len times
        z_lstm_in = z.view(batch_size * num_nodes, 1)
        z_lstm_in = z_lstm_in.unsqueeze(1).repeat(1, seq_len, 1) # (batch * num_nodes, seq_len, 1)
        
        # Apply Bi-LSTM
        out_lstm, _ = self.lstm(z_lstm_in) # (batch * num_nodes, seq_len, 2 * hidden_dim)
        x_gat_in = self.linear(out_lstm) # (batch * num_nodes, seq_len, 16)
        
        # Reshape for GAT: (batch * seq_len * num_nodes, 16)
        x_gat_in = x_gat_in.view(batch_size, num_nodes, seq_len, 16)
        x_gat_in = x_gat_in.transpose(1, 2).contiguous().view(batch_size * seq_len * num_nodes, 16)
        
        num_graphs = batch_size * seq_len
        offsets = torch.arange(num_graphs, device=z.device) * num_nodes
        edge_index_batched = edge_index.unsqueeze(2) + offsets.view(1, 1, -1)
        edge_index_batched = edge_index_batched.view(2, -1)
        
        # Apply GAT
        x_rec = F.relu(self.gat1(x_gat_in, edge_index_batched))
        x_rec = self.gat2(x_rec, edge_index_batched) # (B * seq * N, 22)
        
        x_rec = x_rec.view(batch_size, seq_len, num_nodes, -1)
        return x_rec

class GAL_MAD(nn.Module):
    def __init__(self, in_channels=22, hidden_dim=32):
        super().__init__()
        self.encoder = Encoder(in_channels=in_channels, hidden_dim=hidden_dim)
        self.decoder = Decoder(out_channels=in_channels, hidden_dim=hidden_dim)
        # Register edge_index as buffer so it moves to correct device automatically
        self.register_buffer('edge_index', EDGE_INDEX)
        
    def forward(self, x):
        seq_len = x.shape[1]
        z = self.encoder(x, self.edge_index)
        x_rec = self.decoder(z, seq_len, self.edge_index)
        return x_rec


## 4. Training Loop
Ensure `train.pt` is in the correct path before running.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import time

def train_model():
    # Hyperparameters
    batch_size = 360
    learning_rate = 0.001
    decay_gamma = 0.5
    epochs = 20
    seq_len = 24
    hidden_dim = 32
    
    # Paths
    train_path = 'E:/VHT/AD/train/train.pt'
    
    print("Loading dataset...")
    # Initialize dataset
    train_dataset = RSAnomicDataset(
        data_path=train_path, 
        seq_len=seq_len, 
        is_train=True
    )
    
    # DataLoader
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=0,
        drop_last=False
    )
    
    print(f"Dataset loaded. Total samples: {len(train_dataset)}")
    
    # Device configuration
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Initialize model
    model = GAL_MAD(in_channels=22, hidden_dim=hidden_dim).to(device)
    
    # Loss and optimizer
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # Learning rate scheduler (Decay 0.5 per epoch)
    scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=decay_gamma)
    
    print("Starting training...")
    # Training Loop
    epoch_losses = []
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        start_time = time.time()
        
        for batch_idx, batch_x in enumerate(train_loader):
            batch_x = batch_x.to(device)
            
            # Forward pass
            # Output of GAL_MAD is reconstructed input
            outputs = model(batch_x)
            
            # Compute loss
            loss = criterion(outputs, batch_x)
            
            # Backward and optimize
            optimizer.zero_grad()
            loss.backward()
            
            # Gradient clipping can be useful for LSTM and GNN
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            total_loss += loss.item()
            
            if (batch_idx + 1) % 50 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Step [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}")
                
        # Step the scheduler after each epoch
        scheduler.step()
        
        avg_loss = total_loss / len(train_loader)
        epoch_time = time.time() - start_time
        print(f"Epoch [{epoch+1}/{epochs}] completed in {epoch_time:.2f}s, Average Loss: {avg_loss:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}")
        epoch_losses.append(total_loss/len(train_loader))
        
    print("Training finished.")
    
    # Save the model and the fitted scaler
    print("Saving model and scaler...")
    torch.save(model.state_dict(), 'gal_mad_model.pth')
    
    import pickle
    with open('scaler.pkl', 'wb') as f:
        pickle.dump(train_dataset.scaler, f)
    print("Model and scaler saved successfully.")
    return epoch_losses

if __name__ == '__main__':
    # Add support for Windows multiprocessing
    import multiprocessing
    multiprocessing.freeze_support()
    losses = train_model()

    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, len(losses) + 1), losses, marker='o', linestyle='-', color='b')
    plt.title('Training Loss over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Average Total Loss')
    plt.grid(True)
    plt.show()



## 5. Inference and SHAP Explanation
Ensure `test5.pt` and `train.pt` are in the correct paths.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import shap
import pickle
import json
from torch.utils.data import DataLoader

class LossWrapper(nn.Module):
    """
    Wrapper model to output the Reconstruction MSE Loss for a given input sequence.
    This allows SHAP to attribute the reconstruction error to the input features.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model
        
    def forward(self, x):
        x_rec = self.model(x)
        # Compute MSE loss per sample in the batch
        # x shape: (batch, seq_len, num_nodes, in_channels)
        mse_per_sample = torch.mean((x_rec - x) ** 2, dim=(1, 2, 3))
        return mse_per_sample.unsqueeze(1)

def detect_and_explain():
    # Load hyperparams
    seq_len = 24
    hidden_dim = 32
    c_threshold = 2.0
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Load model
    model = GAL_MAD(in_channels=22, hidden_dim=hidden_dim)
    model.load_state_dict(torch.load('gal_mad_model.pth', map_location=device))
    model.to(device)
    model.eval()
    
    # Load scaler
    with open('scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
        
    # Load test dataset
    print("Loading test data...")
    test_path = 'E:/VHT/AD/test/test5.pt'
    test_dataset = RSAnomicDataset(
        data_path=test_path, 
        seq_len=seq_len, 
        is_train=False,
        scaler=scaler
    )
    
    # Need some background data for GradientExplainer
    # We will load a small subset of the training data as background
    print("Loading background data for SHAP...")
    train_path = 'E:/VHT/AD/train/train.pt'
    train_dataset = RSAnomicDataset(
        data_path=train_path, 
        seq_len=seq_len, 
        is_train=True,
        scaler=scaler
    )
    
    # Take 100 random samples from train data as background
    bg_indices = np.random.choice(len(train_dataset), 100, replace=False)
    background_data = torch.stack([train_dataset[i] for i in bg_indices]).to(device)
    
    loss_model = LossWrapper(model).to(device)
    loss_model.eval()
    
    print("Initializing SHAP GradientExplainer...")
    explainer = shap.GradientExplainer(loss_model, background_data)
    
    # Iterate through test dataset and find anomalies
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    detected_anomalies = 0
    results_json = []
    tp = 0
    fp = 0
    tn = 0
    fn = 0
    
    print("Starting inference and explanation...")
    for idx, (x, y) in enumerate(test_loader):
        x = x.to(device)
        
        # 1. Inference and Anomaly Score
        with torch.no_grad():
            x_rec = model(x)
            mse_loss = torch.mean((x_rec - x) ** 2).item()
            
            # Anomaly score y = Sigmoid(L - c)
            anomaly_score = 1 / (1 + np.exp(-(mse_loss - c_threshold)))
            
        # 2. Check if it's an anomaly (threshold e.g. 0.5 implies mse_loss > c_threshold)
        is_pred_anomaly = anomaly_score > 0.5
        is_true_anomaly = y.sum().item() > 0
        if is_pred_anomaly and is_true_anomaly:
            tp += 1
        elif is_pred_anomaly and not is_true_anomaly:
            fp += 1
        elif not is_pred_anomaly and not is_true_anomaly:
            tn += 1
        else:
            fn += 1

        if is_pred_anomaly:
            detected_anomalies += 1
            print(f"MSE Loss: {mse_loss:.4f}, Anomaly Score: {anomaly_score:.4f}, Ground Truth Label: {y.tolist()}")
            
            # 3. SHAP Explanation
            # Required by GradientExplainer to have requires_grad=True
            x.requires_grad_(True)
            
            # Compute SHAP values
            # Fix for CuDNN RNN backward error in eval mode on GPU
            torch.backends.cudnn.enabled = False
            shap_values = explainer.shap_values(x)
            torch.backends.cudnn.enabled = True
            
            # shap_values is a list for each output. We have 1 output (the loss).
            if isinstance(shap_values, list):
                shap_vals = shap_values[0] # (batch, seq, num_nodes, features)
            else:
                shap_vals = shap_values
                
            # Convert to numpy and drop batch dimension (since batch_size=1)
            shap_vals = np.array(shap_vals)[0] # shape (seq_len, num_nodes, features)
            
            # Absolute SHAP values
            abs_shap = np.abs(shap_vals)
            
            # Aggregate along time axis (sum over seq_len)
            time_aggregated_shap = np.sum(abs_shap, axis=0) # shape (num_nodes, features)
            
            # Weighting: Multiply response_time and container_fs_usage_bytes by 4
            # response_time is index 19
            # container_fs_usage_bytes is index 0
            time_aggregated_shap[:, 19] *= 4
            time_aggregated_shap[:, 0] *= 4
            
            # 4. Service Localization (Sum across features for each node)
            service_scores = np.sum(time_aggregated_shap, axis=1) # shape (num_nodes,)
            anomalous_service_idx = np.argmax(service_scores)
            
            # Map index to service name
            from constants import containers
            anomalous_service_name = containers[anomalous_service_idx]
            
            print(f">> Localized Anomalous Service: {anomalous_service_name} (Index: {anomalous_service_idx})")
            
            # 5. Feature Localization (Max SHAP value within the localized service)
            feature_scores = time_aggregated_shap[anomalous_service_idx] # shape (features,)
            root_cause_metric_idx = np.argmax(feature_scores)
            
            # Map index to feature name
            from constants import cols
            # Features: 0-18 from cols, 19 is rt_sum, 20 is rt_ma24, 21 is rt_ma240
            if root_cause_metric_idx < len(cols):
                root_cause_metric_name = cols[root_cause_metric_idx]
            elif root_cause_metric_idx == 19:
                root_cause_metric_name = "response_time_sum"
            elif root_cause_metric_idx == 20:
                root_cause_metric_name = "response_time_ma_24"
            elif root_cause_metric_idx == 21:
                root_cause_metric_name = "response_time_ma_240"
            else:
                root_cause_metric_name = "Unknown"
                
            print(f">> Localized Root Cause Metric: {root_cause_metric_name} (Index: {root_cause_metric_idx})")
            
            # Save to JSON list
            results_json.append({
                "window_idx": idx,
                "mse_loss": float(mse_loss),
                "anomaly_score": float(anomaly_score),
                "anomalous_service": anomalous_service_name,
                "anomalous_service_idx": int(anomalous_service_idx),
                "root_cause_metric": root_cause_metric_name,
                "root_cause_metric_idx": int(root_cause_metric_idx)
            })
            
            # Break after first anomaly for demonstration purposes, or let it run
            # break
            
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    metrics = {
        "accuracy": float(accuracy),
        "recall": float(recall),
        "specificity": float(specificity),
        "tp": tp, "fp": fp, "tn": tn, "fn": fn
    }
    print(f"\\nTotal Anomalies Detected: {detected_anomalies}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"Specificity: {specificity:.4f}")
    
    # Save JSON to file
    with open('anomalies_report.json', 'w', encoding='utf-8') as f:
        final_output = {'evaluation_metrics': metrics, 'detected_anomalies': results_json}
        json.dump(final_output, f, indent=4, ensure_ascii=False)
    print("Saved anomaly results to anomalies_report.json")

if __name__ == '__main__':
    detect_and_explain()


## 6. Hyperparameter Tuning (PR Curve)
Quét thử nghiệm ngưỡng `c_threshold` từ 0.5 đến 4.0 để vẽ đường cong Precision-Recall, giúp tìm ra điểm giới hạn tối ưu nhất.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import pickle
import os

def plot_pr_curve():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 1. Tải mô hình
    model = GAL_MAD(in_channels=22, hidden_dim=32)
    model.load_state_dict(torch.load('gal_mad_model.pth', map_location=device))
    model.to(device)
    model.eval()
    
    # 2. Tải dữ liệu test
    with open('scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
        
    test_path = 'test/test5.pt' if os.path.exists('test/test5.pt') else 'E:/VHT/AD/test/test5.pt'
    try:
        test_dataset = RSAnomicDataset(data_path=test_path, seq_len=24, stride=16, is_train=False, scaler=scaler)
    except FileNotFoundError:
        print("Vui lòng kiểm tra lại đường dẫn file test5.pt")
        return
        
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    all_mse_loss = []
    all_true_labels = []
    
    print("Đang chạy Inference toàn bộ tập test để trích xuất MSE Loss...")
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            x_rec = model(x)
            
            # Nếu model trả về tuple (ví dụ x_rec, g)
            if isinstance(x_rec, tuple):
                x_rec = x_rec[0]
                
            mse_per_sample = torch.mean((x_rec - x) ** 2, dim=(1,2,3))
            all_mse_loss.extend(mse_per_sample.cpu().numpy())
            
            # y shape có thể là (batch,) chứa 0 hoặc 1
            true_labels = (y.sum(dim=1) > 0).int() if y.dim() > 1 else (y > 0).int()
            all_true_labels.extend(true_labels.cpu().numpy())
            
    all_mse_loss = np.array(all_mse_loss)
    all_true_labels = np.array(all_true_labels)
    
    # 3. Quét c_threshold từ 0.5 đến 4.0
    thresholds = np.arange(0.5, 4.5, 0.5)
    precisions = []
    recalls = []
    f1_scores = []
    
    print(f"\n{'c_thresh':<10} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}")
    print("-" * 50)
    
    best_c = thresholds[0]
    best_f1 = 0
    
    for c in thresholds:
        preds = (all_mse_loss > c).astype(int)
        
        tp = np.sum((preds == 1) & (all_true_labels == 1))
        fp = np.sum((preds == 1) & (all_true_labels == 0))
        fn = np.sum((preds == 0) & (all_true_labels == 1))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        
        if f1 > best_f1:
            best_f1 = f1
            best_c = c
            
        print(f"{c:<10.1f} | {precision:<10.4f} | {recall:<10.4f} | {f1:<10.4f}")
        
    print(f"\n=> Threshold TỐT NHẤT dựa trên F1-Score: c = {best_c:.1f} (F1 = {best_f1:.4f})")
        
    # 4. Vẽ đồ thị PR Curve
    plt.figure(figsize=(10, 6))
    plt.plot(recalls, precisions, marker='o', linestyle='-', color='b', linewidth=2, markersize=8)
    
    # Gắn nhãn c_threshold
    for i, c in enumerate(thresholds):
        plt.annotate(f"c={c:.1f}", (recalls[i], precisions[i]), textcoords="offset points", xytext=(0,10), ha='center')
        
    plt.title('Precision-Recall Curve (Tuning c_threshold)')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.xlim(0, 1.05)
    plt.ylim(0, 1.05)
    plt.show()

if __name__ == '__main__':
    plot_pr_curve()


# MỚI: Tự động rà quét Grid Search tìm ngưỡng c tối ưu
Khối code dưới đây tự động tìm ngưỡng c tốt nhất để đạt F1-Score cao nhất.

In [ ]:
import numpy as np
import torch
import pickle
import os
import warnings
from torch.utils.data import DataLoader

def evaluate_model_grid_search():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = GAL_MAD(in_channels=22, hidden_dim=32)
    model.load_state_dict(torch.load('gal_mad_model.pth', map_location=device))
    model.to(device)
    model.eval()
    
    with open('scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
        
    test_path = 'test/test5.pt' if os.path.exists('test/test5.pt') else 'E:/VHT/AD/test/test5.pt'
    test_dataset = RSAnomicDataset(
        data_path=test_path, 
        seq_len=24, 
        stride=16,
        is_train=False,
        scaler=scaler
    )
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    print("Running inference to collect losses...")
    all_losses = []
    all_labels = []
    
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            y_window = y.max(dim=1)[0].numpy()
            
            x_rec = model(x)
            mse = torch.mean((x_rec - x)**2, dim=(1,2,3)).cpu().numpy()
            
            all_losses.extend(mse)
            all_labels.extend(y_window)
            
    all_losses = np.array(all_losses)
    all_labels = np.array(all_labels)
    
    print("Quét ngưỡng C (Grid Search Thresholding)...")
    best_f1 = 0
    best_c = 0
    best_metrics = {}
    
    min_loss, max_loss = np.min(all_losses), np.max(all_losses)
    c_thresholds = np.linspace(min_loss, max_loss, 100)
    
    warnings.filterwarnings('ignore', category=RuntimeWarning)
    
    for c in c_thresholds:
        anomaly_score = 1 / (1 + np.exp(-np.clip(all_losses - c, -700, 700)))
        preds = (anomaly_score > 0.5).astype(int)
        
        tp = np.sum((preds == 1) & (all_labels == 1))
        fp = np.sum((preds == 1) & (all_labels == 0))
        tn = np.sum((preds == 0) & (all_labels == 0))
        fn = np.sum((preds == 0) & (all_labels == 1))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        if f1 > best_f1:
            best_f1 = f1
            best_c = c
            best_metrics = {
                'Threshold (c)': c,
                'Precision': precision,
                'Recall': recall,
                'F1-Score': f1,
                'Accuracy': (tp + tn) / len(all_labels)
            }
            
    print("\n=== KẾT QUẢ TỐT NHẤT MỚI (GRID SEARCH) ===")
    for k, v in best_metrics.items():
        print(f"{k}: {v:.4f}")
        
    return best_metrics

# Bỏ comment dòng dưới để chạy Grid Search
# evaluate_model_grid_search()



# Trực quan hóa Dữ liệu (Visualization)
Biểu đồ dưới đây sẽ in ra toàn bộ 20 features của 1 Microservice bất kỳ (bạn có thể đổi `node_idx`), đồng thời dải màu nền đỏ (Lỗi thực tế) và màu xanh (Mô hình phát hiện) để so sánh.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import pickle
import os
from torch.utils.data import DataLoader

def plot_microservice_features(node_idx=0, chosen_c=2.0):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = GAL_MAD(in_channels=22, hidden_dim=32)
    model.load_state_dict(torch.load('gal_mad_model.pth', map_location=device))
    model.to(device)
    model.eval()
    
    with open('scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
        
    test_path = 'test/test5.pt' if os.path.exists('test/test5.pt') else 'E:/VHT/AD/test/test5.pt'
    
    # 1. Chạy model để lấy Losses
    test_dataset = RSAnomicDataset(data_path=test_path, is_train=False, scaler=scaler)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    all_losses = []
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            x_rec = model(x)
            mse = torch.mean((x_rec - x)**2, dim=(1,2,3)).cpu().numpy()
            all_losses.extend(mse)
    all_losses = np.array(all_losses)
    
    # 2. Đọc nhãn gốc (Ground Truth) để làm thước đo
    # Chú ý: Dữ liệu lấy trực tiếp từ test5.pt CHƯA hề qua Scaler, đây chính là tín hiệu gốc (Denormalized)
    raw_data = torch.load(test_path)
    L = raw_data.shape[0]
    global_labels = raw_data[:, :, 20].max(dim=1)[0].numpy()
                
    print(f"Đang sử dụng ngưỡng Threshold Cố định c = {chosen_c} để Detect")
    
    # 3. Ánh xạ dự đoán từ dạng Cửa sổ (Window) về lại Dòng thời gian gốc (Timeline)
    anomaly_score = 1 / (1 + np.exp(-np.clip(all_losses - chosen_c, -700, 700)))
    window_preds = (anomaly_score > 0.5).astype(int)
    
    timeline_preds = np.zeros(L)
    for i, pred in enumerate(window_preds):
        if pred == 1:
            start_idx = i * 16
            timeline_preds[start_idx : start_idx + 24] = 1 # Quét mảng cửa sổ thành 1
            
    # 4. Lấy 20 features TÍN HIỆU GỐC của 1 node cụ thể
    features_raw = raw_data[:, node_idx, :20].numpy()
    
    # 5. Bắt đầu vẽ Biểu đồ (Plotting)
    plt.figure(figsize=(16, 6))
    
    # Vẽ 20 đường nét mảnh, mờ đại diện cho 20 metrics (Giá trị Gốc)
    for i in range(20):
        plt.plot(features_raw[:, i], color='gray', alpha=0.5, linewidth=1)
        
    # Tính toán để vẽ vùng đỏ/xanh sao cho phủ kín toàn bộ trục Y dựa trên giá trị của dữ liệu gốc
    y_min, y_max = np.min(features_raw), np.max(features_raw)
    y_range = y_max - y_min
    fill_min = y_min - 0.1 * y_range
    fill_max = y_max + 0.1 * y_range
        
    # Tạo vệt sáng màu ĐỎ cho Lỗi Thực tế (Ground Truth)
    plt.fill_between(range(L), fill_min, fill_max, where=(global_labels == 1), color='red', alpha=0.3, label='Ground Truth (Thực tế)')
    
    # Tạo vệt sáng màu XANH LỤC cho AI phát hiện (Model Detect)
    plt.fill_between(range(L), fill_min, fill_max, where=(timeline_preds == 1), color='lime', alpha=0.4, label='GAL-MAD Detect (AI Dự đoán)')
    
    plt.title(f'Trực quan hóa Dị thường (Tín hiệu Gốc) - Microservice Node [{node_idx}]', fontsize=14, fontweight='bold')
    plt.ylim(fill_min, fill_max) 
    plt.xlabel('Thời gian (Timesteps)', fontsize=12)
    plt.ylabel('Giá trị Đặc trưng (Denormalized Raw Data)', fontsize=12)
    
    # Thu gọn chú thích (Legend) bị trùng lặp
    handles, labels_leg = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels_leg, handles))
    plt.legend(by_label.values(), by_label.keys(), loc='upper right')
    
    plt.tight_layout()
    plt.show()

# Test biểu đồ với Node 0, tự tay cấu hình ngưỡng Threshold c = 1.0 (Có thể thay đổi)
plot_microservice_features(node_idx=0, chosen_c=1.0)
